# Awaaz — Colab Training Runner & Persistence

This notebook manages resilient GPU training for Awaaz Conformer-CTC models on Google Colab:
1. Mounts Google Drive for persistent checkpoints and logs.
2. Untars dataset archive to the local NVMe drive (`/content/data/`) to prevent IO bottlenecks.
3. Inspects GPU hardware and selects optimal `batch_seconds`.
4. Benchmarks throughput and calculates total steps for your target time budget.
5. Launches resumable training with atomic checkpointing (`last.pt`) directly to Google Drive.

In [ ]:
# 1. Mount Google Drive and Setup Environment
from google.colab import drive

drive.mount('/content/drive')

%cd /content
!rm -rf repo
!git clone https://github.com/Paramveersingh-S/Awaaz.git repo
%cd /content/repo
!git checkout main
!pip install -q --upgrade pip
!pip install -e .

In [ ]:
# 2. Untar Dataset from Drive to Fast Local NVMe Disk
# IMPORTANT: Never train directly across Google Drive FUSE; untarring to /content ensures 100% GPU utilization
import os

!mkdir -p /content/data
%cd /content/data

tar_path = '/content/drive/MyDrive/awaaz/data/train_audio.tar'
if os.path.exists(tar_path):
    print('Extracting audio dataset to local disk...')
    !tar -xf /content/drive/MyDrive/awaaz/data/train_audio.tar -C /content/data/
    print('Extraction complete.')
else:
    print(f'Archive not found at {tar_path}. Please run colab_prepare_data.ipynb first or verify path.')

In [ ]:
# 3. Hardware Detection & Batch Seconds Recommendation
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Detected GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)')

    if 'A100' in gpu_name:
        recommended_batch_seconds = 1200
    elif 'L4' in gpu_name:
        recommended_batch_seconds = 600
    elif 'T4' in gpu_name:
        recommended_batch_seconds = 360
    else:
        recommended_batch_seconds = 200
    print(f'Recommended batch_seconds: {recommended_batch_seconds}')
else:
    print('WARNING: Running on CPU runtime! Enable GPU via Runtime -> Change runtime type.')
    recommended_batch_seconds = 80

In [ ]:
# 4. Measured Throughput Benchmark & Budget Step Calculator
# Measures actual audio seconds processed per second over 50 steps
%cd /content/repo

!python -m awaaz.train \
  --config configs/small.yaml \
  --run_dir /content/benchmark_run \
  --manifest_train /content/drive/MyDrive/awaaz/data/train_combined.jsonl \
  --tokenizer /content/drive/MyDrive/awaaz/artifacts/tokenizer.model \
  --cmvn /content/drive/MyDrive/awaaz/artifacts/cmvn.npz \
  --max_steps 50

# Compute steps for a desired Colab session duration (e.g. 10 hours)
import os

import pandas as pd

csv_log = '/content/benchmark_run/train_log.csv'
if os.path.exists(csv_log):
    df = pd.read_csv(csv_log)
    measured_sps = float(df['steps_per_sec'].iloc[-10:].mean())
    budget_hours = 10.0
    target_steps = int(budget_hours * 3600 * measured_sps)
    print(f'Benchmark Steady-State: {measured_sps:.2f} steps/second')
    print(f'Calculated total_steps for {budget_hours}h budget: {target_steps:,} steps')
!rm -rf /content/benchmark_run

In [ ]:
# 5. Launch / Resume Production Training Loop
# Checkpoints and telemetry CSV are written directly to Google Drive.
# TO RESUME AT ANY TIME: Simply re-run this cell.
%cd /content/repo

!python -m awaaz.train \
  --config configs/small.yaml \
  --run_dir /content/drive/MyDrive/awaaz/runs/small_v1 \
  --manifest_train /content/drive/MyDrive/awaaz/data/train_combined.jsonl \
  --manifest_dev /content/drive/MyDrive/awaaz/data/dev_combined.jsonl \
  --tokenizer /content/drive/MyDrive/awaaz/artifacts/tokenizer.model \
  --cmvn /content/drive/MyDrive/awaaz/artifacts/cmvn.npz \
  --resume